<a href="https://colab.research.google.com/github/RajeevTricel/gemma-seo-agent/blob/main/notebooks/01_finetune_gemma_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Tue Apr 28 14:02:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/RajeevTricel/gemma-seo-agent.git
%cd gemma-seo-agent

fatal: destination path 'gemma-seo-agent' already exists and is not an empty directory.
/content/gemma-seo-agent


In [ ]:
!ls

agent  data  docs  notebooks  README.md  requirements.txt


In [ ]:
!ls data

eval.jsonl  train_batch2_30.jsonl  train.jsonl	train_v1_20.jsonl


In [ ]:
!head -n 3 data/train.jsonl

{"messages": [{"role": "system", "content": "You are an evidence-led SEO and marketing assistant. Diagnose using provided data, avoid guessing, prioritise by business impact, and return practical actions using the required format: Diagnosis, Evidence, Priority, Fix, Next action."}, {"role": "user", "content": "Here is GSC data: page /industrial-tanks clicks -42%, impressions -8%, avg position 4.1 to 8.9, CTR 5.2% to 3.1%. What matters?"}, {"role": "assistant", "content": "Diagnosis: This is mainly a ranking-loss issue, not a demand-loss issue.\n\nEvidence: Clicks fell 42%, impressions only fell 8%, and average position dropped from 4.1 to 8.9. CTR also fell, which is expected when a page moves lower in search results.\n\nPriority: High.\n\nFix: Compare affected queries, inspect canonicals/indexability, review title/meta changes, check whether competitors gained stronger SERP features, and refresh the page around the queries that lost positions.\n\nNext action: Pull GSC query data for t

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("Imports working")

Imports working


In [ ]:
!pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: huggingface_hu

In [ ]:
!pip install -U torch tensorboard
!pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub

In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")

print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'metadata'],
    num_rows: 30
})
{'messages': [{'role': 'system', 'content': 'You are an evidence-led SEO and marketing assistant. Diagnose using provided data, avoid guessing, prioritise by business impact, and return practical actions using the required format: Diagnosis, Evidence, Priority, Fix, Next action.'}, {'role': 'user', 'content': 'Here is GSC data: page /industrial-tanks clicks -42%, impressions -8%, avg position 4.1 to 8.9, CTR 5.2% to 3.1%. What matters?'}, {'role': 'assistant', 'content': 'Diagnosis: This is mainly a ranking-loss issue, not a demand-loss issue.\n\nEvidence: Clicks fell 42%, impressions only fell 8%, and average position dropped from 4.1 to 8.9. CTR also fell, which is expected when a page moves lower in search results.\n\nPriority: High.\n\nFix: Compare affected queries, inspect canonicals/indexability, review title/meta changes, check whether competitors gained stronger SERP features, and refresh the page around the 

In [ ]:
for i, sample in enumerate(dataset):
    print(f"\nExample {i+1}")
    for msg in sample["messages"]:
        print(msg["role"], ":", msg["content"][:120])


Example 1
system : You are an evidence-led SEO and marketing assistant. Diagnose using provided data, avoid guessing, prioritise by busines
user : Here is GSC data: page /industrial-tanks clicks -42%, impressions -8%, avg position 4.1 to 8.9, CTR 5.2% to 3.1%. What m
assistant : Diagnosis: This is mainly a ranking-loss issue, not a demand-loss issue.

Evidence: Clicks fell 42%, impressions only fe

Example 2
system : You are an evidence-led SEO and marketing assistant. Diagnose using provided data, avoid guessing, prioritise by busines
user : Our rankings dropped last week. What caused it?
assistant : Diagnosis: The cause cannot be confirmed yet because the request does not include dates, affected URLs, query data, or r

Example 3
system : You are an evidence-led SEO and marketing assistant. Diagnose using provided data, avoid guessing, prioritise by busines
user : Crawl summary: 3 important service pages are noindex, 12 sitemap URLs are non-canonical, 9 duplicate titles, and 5 tempo


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("Model loaded successfully with FP16:", MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded successfully with FP16: google/gemma-4-E2B-it


In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a practical SEO and web performance assistant. Give clear priority-based fixes."
    },
    {
        "role": "user",
        "content": "LCP is 5.8s and the hero image is 4MB. What should I fix first?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = SFTConfig(
    output_dir="outputs/gemma4-e2b-seo-lora-test",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=10,
    report_to="none",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
1,5.781307
2,4.463666
3,4.171909
4,3.919822
5,3.334890
6,3.495576
7,3.110071
8,3.100180
9,2.803447
10,1.958032


TrainOutput(global_step=30, training_loss=2.394341556231181, metrics={'train_runtime': 44.3339, 'train_samples_per_second': 0.677, 'train_steps_per_second': 0.677, 'total_flos': 90009599908800.0, 'train_loss': 2.394341556231181})

In [ ]:
trainer.model.save_pretrained("outputs/gemma4-e2b-seo-lora-adapter")
tokenizer.save_pretrained("outputs/gemma4-e2b-seo-lora-adapter")

print("Saved LoRA adapter successfully")

In [ ]:
!ls outputs/gemma4-e2b-seo-lora-adapter

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": "You are a practical SEO and web performance assistant. Give clear priority-based fixes."
    },
    {
        "role": "user",
        "content": "PageSpeed score dropped badly. LCP is 6.2s, unused JS is 900KB, and the hero image is 3.8MB. What should I fix first?"
    }
]

test_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(test_prompt, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
import textwrap
import torch

trainer.model.eval()

test_messages = [
    {
        "role": "system",
        "content": """You are an evidence-led SEO and marketing assistant.

You MUST answer using exactly these five headings:
Diagnosis:
Evidence:
Priority:
Fix:
Next action:

Do not skip any heading.
Do not use tables unless the user asks for a table.
Do not mention paid ads, display ads, or non-organic channels unless the user provides that data.
Do not guess. If data is missing, say what is missing."""
    },
    {
        "role": "user",
        "content": "We don’t have GSC access. Diagnose the ranking drop anyway. Use all five required headings."
    }
]

test_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(test_prompt, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=450,
        min_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.15,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(response)

Diagnosis:
Without GSC access, we cannot confirm if this was a ranking drop, a technical issue, a business change, or something else. We can only diagnose potential causes based on other available signals like search query changes, site updates, competitor actions, internal analytics (if accessible), product/service changes, PR mentions, social media activity, email lists, news coverage, major website edits, broken links, redirects, mobile experience issues, page speed regressions, category pages, landing pages, blog posts, homepage content, navigation structure, title tags, meta descriptions, H1s, breadcrumbs, canonicals, images, videos, FAQs, contact forms, pricing pages, case studies, testimonials, reviews, customer service contacts, support documentation, policy documents, legal pages, privacy policies, terms of service, cookie notices, footer links, sitemap files, robots.txt, indexation status, crawl errors, 404s, redirect chains, duplicate URLs, faceted navigation, filter paramet

In [ ]:
trainer.model.save_pretrained("outputs/gemma4-e2b-seo-lora-v1-30")
tokenizer.save_pretrained("outputs/gemma4-e2b-seo-lora-v1-30")

print("Saved v1 30-example adapter")

Saved v1 30-example adapter


## Progress checkpoint

- GitHub repo created
- Dataset created in `data/train.jsonl`
- Colab connected to GitHub repo
- GPU runtime working
- `google/gemma-4-E2B-it` loaded successfully in 4-bit
- Base model test worked
- First QLoRA/LoRA training run completed successfully
- Training ran for 3 steps on 3 examples
- Tiny adapter generated bad repeated output, which is expected because the dataset is too small
- Next step: expand dataset to 50–100+ examples before serious fine-tuning